# PS S6E4 - NN digit/log light

目的:
- NN strict baseline に対して、数値列へ軽量な digit/log 特徴を追加する
- CATNUM / TE / high-IV 起点の統計特徴はまだ入れない
- まずは NN が数値表現の軽加工で改善するかを見る

## 1. Imports / config

In [1]:
import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    EXP_NAME = "nn_digit_log_light"
    SAVE_DIR = Path("/kaggle/working/exp_20260406_016_nn_digit_log_light")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    SEED = 42
    N_SPLITS = 5

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 2

    EPOCHS = 30
    BATCH_SIZE = 1024
    LR = 1e-3
    WEIGHT_DECAY = 1e-5
    PATIENCE = 5

    NUM_COLS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    CAT_COLS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Region",
        "Mulching_Used",
    ]

    LIGHT_NUM_TARGETS = [
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    TARGET_CLASSES = ["Low", "Medium", "High"]
    TARGET_TO_IDX = {v: i for i, v in enumerate(TARGET_CLASSES)}
    IDX_TO_TARGET = {i: v for i, v in enumerate(TARGET_CLASSES)}


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.SEED)
print("device:", CFG.DEVICE)
print("save_dir:", CFG.SAVE_DIR)

device: cuda
save_dir: /kaggle/working/exp_20260406_016_nn_digit_log_light


## 2. Load data

In [2]:
INPUT_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")

print(train.shape, test.shape)
display(train.head())
print(train[CFG.TARGET_COL].value_counts(dropna=False))

(630000, 21) (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


## 3. Feature engineering helpers

In [3]:
def safe_log1p(x: pd.Series) -> pd.Series:
    return np.log1p(np.clip(x, a_min=0, a_max=None))


def safe_sqrt(x: pd.Series) -> pd.Series:
    return np.sqrt(np.clip(x, a_min=0, a_max=None))


def build_digit_log_light_features(df: pd.DataFrame, num_targets: list[str]) -> pd.DataFrame:
    out = df.copy()

    for col in num_targets:
        s = out[col].astype(float)

        integer_part = np.floor(s)
        frac_part = s - integer_part

        out[f"{col}__log1p"] = safe_log1p(s)
        out[f"{col}__sqrt"] = safe_sqrt(s)
        out[f"{col}__is_zero"] = (s == 0).astype(np.float32)

        out[f"{col}__integer_part"] = integer_part.astype(np.float32)
        out[f"{col}__frac_part"] = frac_part.astype(np.float32)

        out[f"{col}__first_decimal"] = np.floor(frac_part * 10).astype(np.float32)
        out[f"{col}__second_decimal"] = np.floor((frac_part * 100) % 10).astype(np.float32)

        out[f"{col}__integer_like_flag"] = (np.isclose(frac_part, 0.0)).astype(np.float32)

    return out

## 4. Category / target helpers

In [4]:
def fit_category_maps(train_df, cat_cols):
    cat_maps = {}
    cat_dims = {}
    for col in cat_cols:
        values = train_df[col].astype(str).fillna("__MISSING__")
        uniq = sorted(values.unique().tolist())
        mapping = {v: i + 1 for i, v in enumerate(uniq)}
        cat_maps[col] = mapping
        cat_dims[col] = len(mapping) + 1
    return cat_maps, cat_dims


def transform_categories(df, cat_cols, cat_maps):
    out = pd.DataFrame(index=df.index)
    for col in cat_cols:
        values = df[col].astype(str).fillna("__MISSING__")
        out[col] = values.map(cat_maps[col]).fillna(0).astype(np.int64)
    return out


def transform_target(y):
    return y.map(CFG.TARGET_TO_IDX).astype(np.int64).values


def make_embedding_dim(cardinality: int) -> int:
    return min(32, max(4, int(math.sqrt(cardinality)) + 1))

## 5. Dataset

In [5]:
class TabularDataset(Dataset):
    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X_num)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X_num[idx], self.X_cat[idx]
        return self.X_num[idx], self.X_cat[idx], self.y[idx]

## 6. Model

In [6]:
class TabularMLP(nn.Module):
    def __init__(self, num_dim, cat_cardinalities, n_classes):
        super().__init__()

        self.embeddings = nn.ModuleList()
        emb_total_dim = 0
        for card in cat_cardinalities:
            emb_dim = make_embedding_dim(card)
            self.embeddings.append(nn.Embedding(card, emb_dim))
            emb_total_dim += emb_dim

        in_dim = num_dim + emb_total_dim

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),

            nn.Linear(64, n_classes),
        )

    def forward(self, x_num, x_cat):
        embs = []
        for i, emb in enumerate(self.embeddings):
            embs.append(emb(x_cat[:, i]))
        x = torch.cat([x_num] + embs, dim=1)
        return self.mlp(x)

## 7. Train / predict helpers

In [7]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    loss_fn = nn.CrossEntropyLoss()
    losses = []
    preds = []
    trues = []

    for batch in loader:
        x_num, x_cat, y = batch

        x_num = x_num.to(CFG.DEVICE)
        x_cat = x_cat.to(CFG.DEVICE)
        y = y.to(CFG.DEVICE)

        with torch.set_grad_enabled(is_train):
            logits = model(x_num, x_cat)
            loss = loss_fn(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        losses.append(loss.item())
        preds.append(torch.argmax(logits, dim=1).detach().cpu().numpy())
        trues.append(y.detach().cpu().numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    score = balanced_accuracy_score(trues, preds)

    return float(np.mean(losses)), score, preds, trues


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs = []

    for batch in loader:
        if len(batch) == 3:
            x_num, x_cat, _ = batch
        else:
            x_num, x_cat = batch

        x_num = x_num.to(CFG.DEVICE)
        x_cat = x_cat.to(CFG.DEVICE)

        logits = model(x_num, x_cat)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        probs.append(prob)

    return np.concatenate(probs, axis=0)

## 8. CV training

In [8]:
def fit_nn_cv(train_df, test_df):
    train_feat = build_digit_log_light_features(train_df, CFG.LIGHT_NUM_TARGETS)
    test_feat = build_digit_log_light_features(test_df, CFG.LIGHT_NUM_TARGETS)

    num_feature_cols = [c for c in train_feat.columns if c in CFG.NUM_COLS or "__" in c]

    oof_proba = np.zeros((len(train_df), len(CFG.TARGET_CLASSES)), dtype=np.float32)
    test_proba = np.zeros((len(test_df), len(CFG.TARGET_CLASSES)), dtype=np.float32)
    fold_scores = []

    y_all = train_df[CFG.TARGET_COL].copy()
    X_num_test_base = test_feat[num_feature_cols].copy()
    X_cat_test_base = test_feat[CFG.CAT_COLS].copy()

    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_feat, y_all), 1):
        tr_df = train_feat.iloc[tr_idx].reset_index(drop=True)
        va_df = train_feat.iloc[va_idx].reset_index(drop=True)

        scaler = StandardScaler()
        X_num_tr = scaler.fit_transform(tr_df[num_feature_cols])
        X_num_va = scaler.transform(va_df[num_feature_cols])
        X_num_te = scaler.transform(X_num_test_base)

        cat_maps, cat_dims = fit_category_maps(tr_df, CFG.CAT_COLS)
        X_cat_tr = transform_categories(tr_df, CFG.CAT_COLS, cat_maps).values
        X_cat_va = transform_categories(va_df, CFG.CAT_COLS, cat_maps).values
        X_cat_te = transform_categories(X_cat_test_base, CFG.CAT_COLS, cat_maps).values

        y_tr = transform_target(tr_df[CFG.TARGET_COL])
        y_va = transform_target(va_df[CFG.TARGET_COL])

        train_ds = TabularDataset(X_num_tr, X_cat_tr, y_tr)
        valid_ds = TabularDataset(X_num_va, X_cat_va, y_va)
        test_ds = TabularDataset(X_num_te, X_cat_te, None)

        train_loader = DataLoader(
            train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )
        valid_loader = DataLoader(
            valid_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )
        test_loader = DataLoader(
            test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )

        model = TabularMLP(
            num_dim=len(num_feature_cols),
            cat_cardinalities=[cat_dims[c] for c in CFG.CAT_COLS],
            n_classes=len(CFG.TARGET_CLASSES),
        ).to(CFG.DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.LR,
            weight_decay=CFG.WEIGHT_DECAY,
        )

        best_score = -1.0
        best_state = None
        patience_count = 0

        for epoch in range(1, CFG.EPOCHS + 1):
            train_loss, train_score, _, _ = run_epoch(model, train_loader, optimizer)
            valid_loss, valid_score, _, _ = run_epoch(model, valid_loader, None)

            print(
                f"fold={fold} epoch={epoch} "
                f"train_loss={train_loss:.6f} train_bacc={train_score:.6f} "
                f"valid_loss={valid_loss:.6f} valid_bacc={valid_score:.6f}"
            )

            if valid_score > best_score:
                best_score = valid_score
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= CFG.PATIENCE:
                    print(f"fold={fold} early stopping at epoch={epoch}")
                    break

        model.load_state_dict(best_state)

        valid_proba = predict_proba(model, valid_loader)
        test_fold_proba = predict_proba(model, test_loader)

        oof_proba[va_idx] = valid_proba
        test_proba += test_fold_proba / CFG.N_SPLITS
        fold_scores.append(best_score)

        print(f"fold {fold} best balanced_accuracy = {best_score:.6f}")

        del model, optimizer, scaler, train_loader, valid_loader, test_loader
        del train_ds, valid_ds, test_ds
        gc.collect()
        torch.cuda.empty_cache()

    cv_score = float(np.mean(fold_scores))
    return oof_proba, test_proba, fold_scores, cv_score, num_feature_cols

## 9. Run CV

In [9]:
oof_proba, test_proba, fold_scores, cv_score, num_feature_cols = fit_nn_cv(train, test)

print("fold_scores =", fold_scores)
print("cv_score =", cv_score)
print("n_num_features =", len(num_feature_cols))

fold=1 epoch=1 train_loss=0.252229 train_bacc=0.797054 valid_loss=0.115003 valid_bacc=0.915860
fold=1 epoch=2 train_loss=0.121768 train_bacc=0.907144 valid_loss=0.091004 valid_bacc=0.938764
fold=1 epoch=3 train_loss=0.100320 train_bacc=0.928132 valid_loss=0.081485 valid_bacc=0.954686
fold=1 epoch=4 train_loss=0.089902 train_bacc=0.937682 valid_loss=0.075338 valid_bacc=0.953153
fold=1 epoch=5 train_loss=0.083749 train_bacc=0.943601 valid_loss=0.072563 valid_bacc=0.954099
fold=1 epoch=6 train_loss=0.080092 train_bacc=0.947737 valid_loss=0.070107 valid_bacc=0.958809
fold=1 epoch=7 train_loss=0.077620 train_bacc=0.948999 valid_loss=0.068330 valid_bacc=0.957641
fold=1 epoch=8 train_loss=0.075392 train_bacc=0.951082 valid_loss=0.066513 valid_bacc=0.960132
fold=1 epoch=9 train_loss=0.073835 train_bacc=0.952551 valid_loss=0.067302 valid_bacc=0.958667
fold=1 epoch=10 train_loss=0.072291 train_bacc=0.953763 valid_loss=0.067145 valid_bacc=0.958587
fold=1 epoch=11 train_loss=0.071331 train_bacc=0.

## 10. Save artifacts

In [10]:
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_proba)

oof_pred_idx = oof_proba.argmax(axis=1)
oof_pred_label = pd.Series(oof_pred_idx).map(CFG.IDX_TO_TARGET)

oof_df = pd.DataFrame({
    CFG.ID_COL: train[CFG.ID_COL],
    "y_true": train[CFG.TARGET_COL],
    "y_pred": oof_pred_label,
})
oof_df.to_csv(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_labels.csv", index=False)

sub = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET_COL: pd.Series(test_proba.argmax(axis=1)).map(CFG.IDX_TO_TARGET),
})
sub.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

feature_columns_df = pd.DataFrame({
    "feature": num_feature_cols + CFG.CAT_COLS,
    "type": ["num"] * len(num_feature_cols) + ["cat"] * len(CFG.CAT_COLS),
    "is_engineered_num": [c not in CFG.NUM_COLS for c in num_feature_cols] + [False] * len(CFG.CAT_COLS),
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "cv_score": cv_score,
            "fold_scores": fold_scores,
            "n_raw_num_cols": len(CFG.NUM_COLS),
            "n_final_num_cols": len(num_feature_cols),
            "n_cat_cols": len(CFG.CAT_COLS),
            "n_total_features": len(num_feature_cols) + len(CFG.CAT_COLS),
            "light_num_targets": CFG.LIGHT_NUM_TARGETS,
            "epochs": CFG.EPOCHS,
            "batch_size": CFG.BATCH_SIZE,
            "lr": CFG.LR,
            "weight_decay": CFG.WEIGHT_DECAY,
            "patience": CFG.PATIENCE,
            "device": CFG.DEVICE,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260406_016_nn_digit_log_light


## 11. Quick summary

In [11]:
summary_df = pd.DataFrame({
    "item": [
        "cv_score",
        "n_raw_num_cols",
        "n_final_num_cols",
        "n_cat_cols",
        "n_total_features",
        "epochs",
        "batch_size",
    ],
    "value": [
        cv_score,
        len(CFG.NUM_COLS),
        len(num_feature_cols),
        len(CFG.CAT_COLS),
        len(num_feature_cols) + len(CFG.CAT_COLS),
        CFG.EPOCHS,
        CFG.BATCH_SIZE,
    ],
})
display(summary_df)

,item,value
0,cv_score,0.961227
1,n_raw_num_cols,11.000000
2,n_final_num_cols,75.000000
3,n_cat_cols,8.000000
4,n_total_features,83.000000
5,epochs,30.000000
6,batch_size,1024.000000
